# TensorFlow 2.21 GPU Verification
**Environment:** tf221 | **WSL2 + VS Code**

Run all cells top to bottom. Every cell will print ✅ or ❌ at the end.

---
## Cell 1 — TensorFlow Version & GPU Detection

In [1]:
import tensorflow as tf

print("=== TensorFlow Version ===")
print(f"Version : {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs detected : {gpus}")

if gpus:
    details = tf.config.experimental.get_device_details(gpus[0])
    print(f"GPU Name           : {details.get('device_name', 'N/A')}")
    print(f"Compute Capability : {details.get('compute_capability', 'N/A')}")
    print("\n✅ Cell 1 PASSED")
else:
    print("\n❌ Cell 1 FAILED — No GPU detected")

I0000 00:00:1773456812.783917    2496 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773456813.308548    2496 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773456815.462691    2496 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


=== TensorFlow Version ===
Version : 2.21.0
GPUs detected : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU Name           : NVIDIA RTX A5000 Laptop GPU
Compute Capability : (8, 6)

✅ Cell 1 PASSED


---
## Cell 2 — CUDA & cuDNN Build Info

In [2]:
import tensorflow as tf

print("=== CUDA & cuDNN Build Info ===")
build = tf.sysconfig.get_build_info()

print(f"CUDA Version         : {build.get('cuda_version', 'N/A')}")
print(f"cuDNN Version        : {build.get('cudnn_version', 'N/A')}")
print(f"Is CUDA Build        : {build.get('is_cuda_build', False)}")
print(f"CUDA Compute Caps    : {build.get('cuda_compute_capabilities', 'N/A')}")

if build.get('is_cuda_build'):
    print("\n✅ Cell 2 PASSED")
else:
    print("\n❌ Cell 2 FAILED — Not a CUDA build")

=== CUDA & cuDNN Build Info ===
CUDA Version         : 12.5.1
cuDNN Version        : 9
Is CUDA Build        : True
CUDA Compute Caps    : ['sm_60', 'sm_70', 'sm_80', 'sm_89', 'compute_90']

✅ Cell 2 PASSED


---
## Cell 3 — Basic GPU Matmul Test

In [3]:
import tensorflow as tf
import time

print("=== Basic GPU Matmul [2000x2000] ===")

try:
    with tf.device('/GPU:0'):
        a = tf.random.normal([2000, 2000])
        b = tf.random.normal([2000, 2000])
        start = time.time()
        c = tf.matmul(a, b)
        _ = c.numpy()
        elapsed = time.time() - start
    print(f"Result shape : {c.shape}")
    print(f"Time         : {elapsed:.4f} seconds")
    print("\n✅ Cell 3 PASSED")
except Exception as e:
    print(f"\n❌ Cell 3 FAILED — {e}")

=== Basic GPU Matmul [2000x2000] ===


I0000 00:00:1773456819.205520    2496 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13713 MB memory:  -> device: 0, name: NVIDIA RTX A5000 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Result shape : (2000, 2000)
Time         : 0.0948 seconds

✅ Cell 3 PASSED


---
## Cell 4 — Heavy GPU Stress Test [10000x10000]

In [4]:
import tensorflow as tf
import time

print("=== Heavy GPU Stress Test [10000x10000] ===")
print("This pushes your GPU hard — expect a few seconds...")

try:
    with tf.device('/GPU:0'):
        a = tf.random.normal([10000, 10000])
        b = tf.random.normal([10000, 10000])
        start = time.time()
        c = tf.matmul(a, b)
        _ = c.numpy()
        elapsed = time.time() - start
    print(f"Result shape : {c.shape}")
    print(f"Time         : {elapsed:.4f} seconds")
    print("\n✅ Cell 4 PASSED")
except Exception as e:
    print(f"\n❌ Cell 4 FAILED — {e}")

=== Heavy GPU Stress Test [10000x10000] ===
This pushes your GPU hard — expect a few seconds...
Result shape : (10000, 10000)
Time         : 0.5111 seconds

✅ Cell 4 PASSED


---
## Cell 5 — GPU vs CPU Speed Comparison

In [5]:
import tensorflow as tf
import time

print("=== GPU vs CPU Speed Comparison [5000x5000] ===")

a = tf.random.normal([5000, 5000])
b = tf.random.normal([5000, 5000])

# CPU
with tf.device('/CPU:0'):
    a_cpu = tf.identity(a)
    b_cpu = tf.identity(b)
    start = time.time()
    c_cpu = tf.matmul(a_cpu, b_cpu)
    _ = c_cpu.numpy()
    cpu_time = time.time() - start

# GPU
with tf.device('/GPU:0'):
    a_gpu = tf.identity(a)
    b_gpu = tf.identity(b)
    start = time.time()
    c_gpu = tf.matmul(a_gpu, b_gpu)
    _ = c_gpu.numpy()
    gpu_time = time.time() - start

print(f"CPU Time : {cpu_time:.4f} seconds")
print(f"GPU Time : {gpu_time:.4f} seconds")
print(f"Speedup  : {cpu_time / gpu_time:.1f}x faster on GPU")
print("\n✅ Cell 5 PASSED")

=== GPU vs CPU Speed Comparison [5000x5000] ===
CPU Time : 0.4935 seconds
GPU Time : 0.0858 seconds
Speedup  : 5.8x faster on GPU

✅ Cell 5 PASSED


---
## Cell 6 — Real Neural Network Training on GPU

In [6]:
import tensorflow as tf
import numpy as np
import time

print("=== Neural Network Training Test ===")
print("Building and training a simple model on GPU...")

try:
    with tf.device('/GPU:0'):
        # Dummy dataset
        X = np.random.randn(10000, 128).astype(np.float32)
        y = np.random.randint(0, 10, size=(10000,))
        y = tf.keras.utils.to_categorical(y, num_classes=10)

        # Model — using Input layer to avoid Keras 3 warning
        inputs = tf.keras.Input(shape=(128,))
        x = tf.keras.layers.Dense(512, activation='relu')(inputs)
        x = tf.keras.layers.Dense(256, activation='relu')(x)
        x = tf.keras.layers.Dense(128, activation='relu')(x)
        outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
        model = tf.keras.Model(inputs=inputs, outputs=outputs)

        model.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        start = time.time()
        history = model.fit(X, y, epochs=5, batch_size=256, verbose=1)
        elapsed = time.time() - start

    print(f"\nTotal training time : {elapsed:.2f} seconds")
    print(f"Final loss          : {history.history['loss'][-1]:.4f}")
    print(f"Final accuracy      : {history.history['accuracy'][-1]:.4f}")
    print("\n✅ Cell 6 PASSED")
except Exception as e:
    print(f"\n❌ Cell 6 FAILED — {e}")

=== Neural Network Training Test ===
Building and training a simple model on GPU...
Epoch 1/5


I0000 00:00:1773456822.612591    2601 service.cc:153] XLA service 0x7884bc043a60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773456822.612631    2601 service.cc:161]   StreamExecutor [0]: NVIDIA RTX A5000 Laptop GPU, Compute Capability 8.6 (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.20.0)
I0000 00:00:1773456822.651737    2601 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1773456822.861064    2601 cuda_dnn.cc:461] Loaded cuDNN version 92000
I0000 00:00:1773456822.874834    2601 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1672__.18
I0000 00:00:1773456823.650265    2601 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.

23/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0995 - loss: 2.3477 

I0000 00:00:1773456826.445304    2601 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
I0000 00:00:1773456826.666537    2599 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1672__.18


40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - accuracy: 0.1013 - loss: 2.3204
Epoch 2/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1757 - loss: 2.2483
Epoch 3/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2271 - loss: 2.1695
Epoch 4/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2952 - loss: 2.0371
Epoch 5/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3820 - loss: 1.8623

Total training time : 7.66 seconds
Final loss          : 1.8623
Final accuracy      : 0.3820

✅ Cell 6 PASSED


---
## Cell 7 — GPU Memory Usage

In [7]:
import tensorflow as tf
import subprocess

print("=== GPU Memory Usage ===")

try:
    result = subprocess.run(
        ['nvidia-smi',
         '--query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu',
         '--format=csv,noheader,nounits'],
        capture_output=True, text=True
    )
    lines = result.stdout.strip().split('\n')
    for line in lines:
        parts = [p.strip() for p in line.split(',')]
        print(f"GPU Name     : {parts[0]}")
        print(f"Total Memory : {parts[1]} MB")
        print(f"Used Memory  : {parts[2]} MB")
        print(f"Free Memory  : {parts[3]} MB")
        print(f"GPU Util     : {parts[4]}%")
    print("\n✅ Cell 7 PASSED")
except Exception as e:
    print(f"\n❌ Cell 7 FAILED — {e}")

=== GPU Memory Usage ===
GPU Name     : NVIDIA RTX A5000 Laptop GPU
Total Memory : 16384 MB
Used Memory  : 13887 MB
Free Memory  : 2289 MB
GPU Util     : 15%

✅ Cell 7 PASSED


---
## Cell 8 — Full Summary

In [8]:
import tensorflow as tf

print("=== Final Summary ===")
print(f"TensorFlow Version : {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
build = tf.sysconfig.get_build_info()

if gpus:
    details = tf.config.experimental.get_device_details(gpus[0])
    print(f"GPU               : {details.get('device_name', 'N/A')}")
    print(f"Compute Cap       : {details.get('compute_capability', 'N/A')}")

print(f"CUDA Version      : {build.get('cuda_version', 'N/A')}")
print(f"cuDNN Version     : {build.get('cudnn_version', 'N/A')}")
print(f"CUDA Build        : {build.get('is_cuda_build', False)}")
print("")
print("✅ All checks complete. Your GPU setup is working correctly.")

=== Final Summary ===
TensorFlow Version : 2.21.0
GPU               : NVIDIA RTX A5000 Laptop GPU
Compute Cap       : (8, 6)
CUDA Version      : 12.5.1
cuDNN Version     : 9
CUDA Build        : True

✅ All checks complete. Your GPU setup is working correctly.
